# Ejercicio 4: Construcción de modelos de Machine Learning

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from xgboost import XGBClassifier

from src.config import OUT_DIR

plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_columns", None)
RANDOM_STATE = 42


## 4.0 Carga de datos, variable respuesta y variables predictoras

In [ ]:
df = pd.read_parquet(OUT_DIR / "ml_dataset.parquet")
print(f"Dataset cargado: {len(df):,} observaciones, {df.shape[1]} columnas")

UMBRAL_ALTA_CIANOBACTERIA = 10.0
df["alta_cianobacteria"] = (df["cianobacteria"] > UMBRAL_ALTA_CIANOBACTERIA).astype(int)

df["alta_cianobacteria"].value_counts(normalize=True).rename("proporcion")


Dataset cargado: 538,675 observaciones, 14 columnas


alta_cianobacteria
0    0.926341
1    0.073659
Name: proporcion, dtype: float64

In [ ]:
dia_anio = df["fecha"].dt.dayofyear
df["dia_anio_sin"] = np.sin(2 * np.pi * dia_anio / 365)
df["dia_anio_cos"] = np.cos(2 * np.pi * dia_anio / 365)
df["ndvi_x_ndwi"] = df["ndvi"] * df["ndwi"]

df[["fecha", "dia_anio_sin", "dia_anio_cos", "ndvi", "ndwi", "ndvi_x_ndwi"]].head(3)


,fecha,dia_anio_sin,dia_anio_cos,ndvi,ndwi,ndvi_x_ndwi
0,2025-01-18,0.304921,0.952378,-0.411765,0.900990,-0.370996
1,2025-01-18,0.304921,0.952378,0.734375,0.086420,0.063465
2,2025-01-18,0.304921,0.952378,-0.127273,0.659574,-0.083946


### Conjunto final de variables predictoras

| Variable | Tipo | Justificación |
|---|---|---|
| `green` | Banda espectral | No interviene en la fórmula de NDCI/cianobacteria; aporta señal de pigmentos algales (cercana al pico de reflectancia de clorofila). |
| `nir` | Banda espectral | Insumo de NDVI/NDWI; contraste agua/vegetación/biomasa flotante. |
| `ndvi` | Índice espectral | Biomasa fotosintética superficial; correlación moderada y biológicamente esperable con la respuesta (r ≈ 0.21, Ejercicio 2.5). |
| `ndwi` | Índice espectral | Delimita el cuerpo de agua; no interviene en la fórmula de la respuesta. |
| `lat`, `lon` | Espacial | Capturan gradientes espaciales (zonas costeras, afluentes, urbanización) dentro de cada lago. |
| `lago` | Espacial (categórica) | Diferencias estructurales/tróficas entre Atitlán y Amatitlán (ver Ejercicio 2.3). Se excluirá en el Ejercicio 7 (generalización entre lagos), donde por diseño el modelo no puede ver la etiqueta del lago de evaluación. |
| `dia_anio_sin`, `dia_anio_cos` | Temporal (derivada) | Estacionalidad de las floraciones, sin discontinuidad de fin de año. |
| `ndvi_x_ndwi` | Derivada (interacción) | Resalta píxeles de agua con señal de biomasa simultánea. |


In [ ]:
NUM_FEATURES = ["green", "nir", "ndvi", "ndwi", "lat", "lon",
                 "dia_anio_sin", "dia_anio_cos", "ndvi_x_ndwi"]
CAT_FEATURES = ["lago"]
FEATURES = NUM_FEATURES + CAT_FEATURES
TARGET = "alta_cianobacteria"

X = df[FEATURES].copy()
y = df[TARGET].copy()
X.head(3)


,green,nir,ndvi,ndwi,lat,lon,dia_anio_sin,dia_anio_cos,ndvi_x_ndwi,lago
0,0.0096,0.0005,-0.411765,0.900990,14.747720,-91.168902,0.304921,0.952378,-0.370996,atitlan
1,0.0132,0.0111,0.734375,0.086420,14.747712,-91.167974,0.304921,0.952378,0.063465,atitlan
2,0.0117,0.0024,-0.127273,0.659574,14.746790,-91.165660,0.304921,0.952378,-0.083946,atitlan


## 4.2 División entrenamiento / prueba (70 % / 30 %)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)

print(f"Entrenamiento: {len(X_train):,} obs. | % clase positiva: {y_train.mean()*100:.2f}%")
print(f"Prueba:        {len(X_test):,} obs. | % clase positiva: {y_test.mean()*100:.2f}%")


Entrenamiento: 377,072 obs. | % clase positiva: 7.37%
Prueba:        161,603 obs. | % clase positiva: 7.37%


## Preprocesamiento


In [ ]:
preprocesador_lineal = ColumnTransformer([
    ("num", StandardScaler(), NUM_FEATURES),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), CAT_FEATURES),
])

preprocesador_arboles = ColumnTransformer([
    ("num", "passthrough", NUM_FEATURES),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), CAT_FEATURES),
])

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)


### Modelo 1: Regresión Logística

In [ ]:
lr_pipe = Pipeline([
    ("pre", preprocesador_lineal),
    ("clf", LogisticRegression(max_iter=5000, class_weight="balanced",
                                random_state=RANDOM_STATE)),
])

lr_param_grid = {
    "clf__C": [0.001, 0.01, 0.1, 1, 10, 100],
}

lr_search = RandomizedSearchCV(
    lr_pipe, lr_param_grid, n_iter=6, cv=cv, scoring="average_precision",
    n_jobs=-1, random_state=RANDOM_STATE, refit=True,
)
lr_search.fit(X_train, y_train)

print("Mejor C:", lr_search.best_params_)
print(f"Average Precision (CV): {lr_search.best_score_:.4f}")
modelo_lr = lr_search.best_estimator_


Mejor C: {'clf__C': 1}
Average Precision (CV): 0.6933


**Hiperparámetro ajustado:** C (inverso de la fuerza de regularización L2), en una
grilla logarítmica de 0.001 a 100. Es el único hiperparámetro relevante de una
regresión logística estándar; controla el sobreajuste penalizando coeficientes
grandes. Con solo 9 predictoras numéricas + 1 dummy, el riesgo de sobreajuste es bajo,
por lo que se espera que el valor óptimo de C no sea extremo.


### Modelo 2: Random Forest

In [ ]:
rf_pipe = Pipeline([
    ("pre", preprocesador_arboles),
    ("clf", RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE,
                                    n_jobs=-1)),
])

rf_param_grid = {
    "clf__n_estimators": [200, 400],
    "clf__max_depth": [8, 14, 20, None],
    "clf__min_samples_leaf": [1, 5, 20],
    "clf__max_features": ["sqrt", 0.5],
}

rf_search = RandomizedSearchCV(
    rf_pipe, rf_param_grid, n_iter=8, cv=cv, scoring="average_precision",
    n_jobs=1, random_state=RANDOM_STATE, refit=True,
)
rf_search.fit(X_train, y_train)

print("Mejores hiperparámetros:", rf_search.best_params_)
print(f"Average Precision (CV): {rf_search.best_score_:.4f}")
modelo_rf = rf_search.best_estimator_


Mejores hiperparámetros: {'clf__n_estimators': 400, 'clf__min_samples_leaf': 5, 'clf__max_features': 'sqrt', 'clf__max_depth': 20}
Average Precision (CV): 0.9157


**Hiperparámetros ajustados:**

- n_estimators (200 vs. 400 árboles): más árboles reducen la varianza del ensamble
  hasta un punto de rendimientos decrecientes; se evalúan dos valores para verificar
  si vale la pena el costo computacional adicional.
- max_depth (8, 14, 20, sin límite): controla directamente el sobreajuste; árboles
  muy profundos pueden memorizar ruido de píxeles individuales.
- min_samples_leaf (1, 5, 20): exige un número mínimo de observaciones por hoja,
  suavizando las fronteras de decisión y reduciendo sensibilidad a outliers
  espectrales (nubes residuales, artefactos).
- max_features (sqrt vs. 50 % de las variables): cuántas variables candidatas se
  evalúan en cada split; afecta la correlación entre árboles del bosque.


### Modelo 3: Gradient Boosting (XGBoost)

In [ ]:
n_neg, n_pos = np.bincount(y_train)
scale_pos_weight = n_neg / n_pos

xgb_pipe = Pipeline([
    ("pre", preprocesador_arboles),
    ("clf", XGBClassifier(
        tree_method="hist", eval_metric="aucpr", n_estimators=300,
        scale_pos_weight=scale_pos_weight, random_state=RANDOM_STATE, n_jobs=-1,
    )),
])

xgb_param_grid = {
    "clf__max_depth": [3, 4, 6, 8],
    "clf__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "clf__subsample": [0.7, 0.9, 1.0],
    "clf__colsample_bytree": [0.7, 0.9, 1.0],
}

xgb_search = RandomizedSearchCV(
    xgb_pipe, xgb_param_grid, n_iter=8, cv=cv, scoring="average_precision",
    n_jobs=1, random_state=RANDOM_STATE, refit=True,
)
xgb_search.fit(X_train, y_train)

print("Mejores hiperparámetros:", xgb_search.best_params_)
print(f"Average Precision (CV): {xgb_search.best_score_:.4f}")
modelo_xgb = xgb_search.best_estimator_


Mejores hiperparámetros: {'clf__subsample': 0.9, 'clf__max_depth': 8, 'clf__learning_rate': 0.1, 'clf__colsample_bytree': 0.9}
Average Precision (CV): 0.9173


**Hiperparámetros ajustados:**

- max_depth (3-8): profundidad de cada árbol individual; en *boosting* árboles
  poco profundos ("weak learners") suelen generalizar mejor que en Random Forest.
- learning_rate (0.01-0.2): tasa de aprendizaje; valores bajos requieren más
  árboles pero suelen generalizar mejor.
- subsample y colsample_bytree (0.7-1.0): fracción de observaciones y de
  variables muestreadas por árbol; añaden aleatoriedad tipo bagging dentro del
  boosting, reduciendo sobreajuste.
- scale_pos_weight se fija (no se ajusta por búsqueda) al cociente exacto
  clase-negativa/clase-positiva del conjunto de entrenamiento, el valor recomendado
  por la documentación de XGBoost para clasificación binaria desbalanceada.


## 4.3 Resumen de la búsqueda de hiperparámetros


In [ ]:
resumen_busqueda = pd.DataFrame([
    {"modelo": "Regresión Logística", "mejor_avg_precision_cv": lr_search.best_score_,
     "mejores_hiperparametros": lr_search.best_params_},
    {"modelo": "Random Forest", "mejor_avg_precision_cv": rf_search.best_score_,
     "mejores_hiperparametros": rf_search.best_params_},
    {"modelo": "XGBoost", "mejor_avg_precision_cv": xgb_search.best_score_,
     "mejores_hiperparametros": xgb_search.best_params_},
])
resumen_busqueda


,modelo,mejor_avg_precision_cv,mejores_hiperparametros
0,Regresión Logística,0.693336,{'clf__C': 1}
1,Random Forest,0.915672,"{'clf__n_estimators': 400, 'clf__min_samples_l..."
2,XGBoost,0.917276,"{'clf__subsample': 0.9, 'clf__max_depth': 8, '..."


## 4.4 Persistencia de modelos y partición

In [ ]:
MODELOS_DIR = OUT_DIR / "modelos"
MODELOS_DIR.mkdir(exist_ok=True)

joblib.dump(modelo_lr, MODELOS_DIR / "modelo_logreg.joblib")
joblib.dump(modelo_rf, MODELOS_DIR / "modelo_rf.joblib")
joblib.dump(modelo_xgb, MODELOS_DIR / "modelo_xgb.joblib")

split_info = pd.DataFrame({
    "idx_original": X_train.index.tolist() + X_test.index.tolist(),
    "particion": ["train"] * len(X_train) + ["test"] * len(X_test),
})
split_info.to_parquet(OUT_DIR / "split_train_test.parquet", index=False)

print("Modelos guardados en", MODELOS_DIR)
print("Partición 70/30 guardada en", OUT_DIR / "split_train_test.parquet")


Modelos guardados en /Users/nilsmurallesmorales/universidad/octavosemestre/data/lab42/data/processed/modelos
Partición 70/30 guardada en /Users/nilsmurallesmorales/universidad/octavosemestre/data/lab42/data/processed/split_train_test.parquet
